# 🧠 CASCADES v9 Pro: One-Click Reproduction

**Prove that your 4B heretic model can learn continuously without forgetting — in ~5 minutes on a free T4 GPU.**

This notebook reproduces the core result from [CASCADES v9 Pro](https://github.com/Bender1011001/CASCADES--continual-PEFT-for-Local-LLMs):

| Method | Model | Backward Transfer | Result |
| --- | --- | --- | --- |
| Standard LoRA | Qwen3-8B Heretic | **−12.18%** | 💀 Catastrophic forgetting |
| **CASCADES v9 Pro** | **Qwen3-4B Heretic** | **+0.82%** | ✅ Zero forgetting |

CASCADES constrains adapter updates on a **Stiefel manifold** (Riemannian geometry) so new learning physically cannot overwrite old knowledge.

**Expected runtime:** ~5 minutes on a T4 GPU.

📄 [Paper](https://github.com/Bender1011001/CASCADES--continual-PEFT-for-Local-LLMs/blob/main/papers/CASCADES_v9_Final_Paper.md) | 📖 [Plain Language Explainer](https://github.com/Bender1011001/CASCADES--continual-PEFT-for-Local-LLMs/blob/main/papers/CASCADES_v9_Plain_Language.md)

## Step 1: Clone Repo & Install Dependencies

This installs PyTorch, HuggingFace Transformers, bitsandbytes (for 4-bit quantization), and all other requirements.

In [ ]:
!git clone https://github.com/Bender1011001/CASCADES--continual-PEFT-for-Local-LLMs.git
%cd CASCADES--continual-PEFT-for-Local-LLMs
!pip install -q -r requirements.txt

## Step 2: Verify GPU

You need a GPU with at least 8GB VRAM. Colab's free T4 (16GB) works perfectly.

In [ ]:
!nvidia-smi

import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("⚠️ No GPU detected! Go to Runtime → Change runtime type → T4 GPU")

## Step 3: Run the Reproduction

This does the following:
1. Loads **Qwen3-4B Heretic** in 4-bit NF4 quantization (fits in ~5GB VRAM)
2. Injects CASCADES v9 adapters (Stiefel manifold + Liquid Core routing)
3. Trains on **Task 0 (Logic reasoning)** → evaluates
4. Trains on **Task 1 (Decomposition reasoning)** → evaluates both tasks
5. Computes **Backward Transfer (BWT)**: did learning Task 1 destroy Task 0?

**The key result:** If BWT ≥ 0, the model preserved its old knowledge perfectly.

In [ ]:
import sys
import os

# Ensure the repo root is on the Python path
repo_root = "/content/CASCADES--continual-PEFT-for-Local-LLMs"
sys.path.insert(0, repo_root)

# Change to cascades_exp so data files are found
os.chdir(os.path.join(repo_root, "cascades_exp"))

from reproduce_the_breakthrough import reproduce_breakthrough

reproduce_breakthrough()

## 📊 Interpreting Results

The key metric printed above is **Backward Transfer (BWT)**:

- **BWT > 0** → ✅ The model **improved** on Task 0 after learning Task 1 (positive transfer)
- **BWT = 0** → The model perfectly preserved Task 0 knowledge
- **BWT < 0** → ❌ The model **forgot** Task 0 (catastrophic forgetting)

For comparison:
- Standard LoRA on Qwen3-8B Heretic: **BWT = −12.18%** (destroyed)
- CASCADES v9 Pro (full run): **BWT = +0.82%** (improved!)

This lightning reproduction uses only 10 gradient steps per task (vs. the full 45 in the paper), so the absolute accuracy numbers will be lower. **The key signal is the BWT sign** — if it's non-negative, CASCADES did its job.

### Why this matters for heretic/abliterated models

Abliterated models have had their safety guardrails surgically removed, which makes them extremely fragile under standard fine-tuning. Standard LoRA literally destroys their reasoning architecture. CASCADES constrains all adapter updates to a Stiefel manifold (an orthogonality surface), making it mathematically impossible for new learning to interfere with established knowledge.

## 🔗 Links

- **GitHub**: [Bender1011001/CASCADES--continual-PEFT-for-Local-LLMs](https://github.com/Bender1011001/CASCADES--continual-PEFT-for-Local-LLMs)
- **Research Paper**: [CASCADES_v9_Final_Paper.md](https://github.com/Bender1011001/CASCADES--continual-PEFT-for-Local-LLMs/blob/main/papers/CASCADES_v9_Final_Paper.md)
- **Plain Language Explainer**: [CASCADES_v9_Plain_Language.md](https://github.com/Bender1011001/CASCADES--continual-PEFT-for-Local-LLMs/blob/main/papers/CASCADES_v9_Plain_Language.md)
- **Version History Analysis**: [cascades_v4_v9_analysis.md](https://github.com/Bender1011001/CASCADES--continual-PEFT-for-Local-LLMs/blob/main/cascades_v4_v9_analysis.md)

---
*CASCADES is open-source research for the unconstrained reasoning community.*